In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
import sys
from pathlib import Path

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "training" / "data_utils.py").is_file()
)
sys.path.insert(0, str(repo_root / "src"))
from training.data_utils import grouped_track_id_split


In [ ]:
og_df = pd.read_csv("../../../data/SpotGenTrack/Data Sources/spotify_tracks.csv", index_col=0)
low_df = pd.read_csv("../../../data/SpotGenTrack/Features Extracted/low_level_audio_features.csv", index_col=0)

df = pd.merge(
    low_df,
    og_df,
    left_on="track_id",
    right_on="id",
    how="inner"
)

df = df.dropna()
df = df.drop_duplicates()
drop_cols = [
    "track_id",
    "album_id",
    "analysis_url",
    "artists_id",
    "available_markets",
    "country",
    "href",
    "lyrics",
    "name",
    "playlist",
    "preview_url",
    "track_href",
    "track_name_prev",
    "type",
    "uri",
    "disc_number",
    "track_number"
]

df = df.drop(columns=drop_cols, errors="ignore")

In [ ]:
target = "popularity"

track_ids = df["id"].astype(str)
X = df.drop(columns=[target, "id"])
y = df[target]

train_ids, validation_ids, test_ids = map(
    set, grouped_track_id_split(track_ids.tolist())
)
train_mask = track_ids.isin(train_ids)
validation_mask = track_ids.isin(validation_ids)
test_mask = track_ids.isin(test_ids)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[validation_mask], y.loc[validation_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


In [ ]:
knn = KNeighborsRegressor(n_neighbors=30)
knn.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = knn.predict(X_train_scaled)
y_val_pred = knn.predict(X_val_scaled)
y_test_pred = knn.predict(X_test_scaled)

# R²
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)

# RMSE
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

# MAE
train_mae = mean_absolute_error(y_train, y_train_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print(f"Train R2: {train_r2:.4f}")
print(f"Validation R2: {val_r2:.4f}")
print(f"Test R2: {test_r2:.4f}")

print(f"Train RMSE: {train_rmse:.4f}")
print(f"Validation RMSE: {val_rmse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")

print(f"Train MAE: {train_mae:.4f}")
print(f"Validation MAE: {val_mae:.4f}")
print(f"Test MAE: {test_mae:.4f}")

In [ ]:
k_values = [1, 3, 5, 10, 20, 30, 50]

for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)

    val_r2 = r2_score(y_val, knn.predict(X_val_scaled))
    print(f"k = {k:2d} | Validation R2 = {val_r2:.4f}")